[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Xrenya/stable_diffusion/blob/main/SD2.ipynb)

# Stable Diffusion 2.x U-Net from Scratch

This tutorial builds the **U-Net** (noise prediction network) used in
**Stable Diffusion 2.x** from scratch in PyTorch, then loads the official
pretrained weights from HuggingFace.

## SD 2.x vs SD 1.5 - What Changed?

| Aspect | SD 1.5 | SD 2.x |
|--------|--------|--------|
| Text encoder | CLIP ViT-L/14 (768-dim) | OpenCLIP ViT-H/14 (**1024-dim**) |
| Native resolution | 512×512 (64×64 latent) | 768×768 (**96×96 latent**) |
| Cross-attention dim | 768 | **1024** |
| Attention head dim | Fixed (40/80/160) | **Variable** (5/10/20/20) via config |
| Projection in attention | Conv2d 1×1 | **Linear** (use_linear_projection=True) |
| Architecture style | Hardcoded layer counts | **Config-driven** (dataclass) |
| Weight format | Single `.ckpt` blob | **Diffusers** (per-component safetensors) |

The core U-Net architecture is the same encoder-bottleneck-decoder with skip
connections, but SD 2.x uses a config-driven design (like HuggingFace diffusers)
instead of hardcoded layer definitions.

## What We Will Build

This tutorial focuses on the **U-Net** component only (the diffusion backbone).
The overall SD 2.x pipeline is the same as SD 1.5:

```
Text => OpenCLIP (1024-dim) => U-Net (iterative denoising) => VAE Decoder => Image
```

| Component | Purpose |
|---|---|
| `UNet2DConditionConfig` | Dataclass holding all architecture hyperparameters |
| `ResnetBlock2D` | Residual block with timestep conditioning |
| `CrossAttention` | Unified self/cross-attention (context=None => self-attn) |
| `BasicTransformerBlock` | Self-attn + Cross-attn + GeGLU FFN |
| `Transformer2DModel` | Wraps transformer blocks with spatial reshape |
| `CrossAttnDownBlock2D` / `CrossAttnUpBlock2D` | Encoder/decoder blocks with attention |
| `UNet2DConditionModel` | The full U-Net assembled from config |

## 0. Dependencies

In [ ]:
!pip install torch safetensors huggingface_hub diffusers==0.29.2 transformers==4.44.2
# current version of huggingface is a little different

## 1. Configuration

SD 2.x uses a **config-driven** architecture: all hyperparameters live in a
dataclass rather than being hardcoded. This is the same approach used by
HuggingFace diffusers.

Key differences from SD 1.5:
- `sample_size=96` (not 64) - corresponds to 768×768 pixel images
- `cross_attention_dim=1024` (not 768) - matches OpenCLIP ViT-H output
- `attention_head_dim=(5, 10, 20, 20)` - variable per level (SD 1.5 used fixed sizes)
- `use_linear_projection=True` - Linear layers instead of 1×1 convolutions
  in the Transformer2DModel (slightly more efficient)

In [ ]:
import json
import math
import os
from dataclasses import dataclass
from typing import Optional, List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file


@dataclass
class UNet2DConditionConfig:
    """
    All hyperparameters for the SD 2.x U-Net in one place.

    This mirrors the config.json shipped with diffusers checkpoints.
    You can load it from HuggingFace or define it manually.
    """
    sample_size: int = 96              # Latent spatial size (96 for SD 2.x, 64 for SD 1.5)
    in_channels: int = 4               # Latent channels from VAE
    out_channels: int = 4              # Predicted noise has same shape as input
    center_input_sample: bool = False
    flip_sin_to_cos: bool = True       # Sinusoidal embedding order
    freq_shift: int = 0

    # Block types for encoder and decoder
    # "CrossAttnDownBlock2D" = ResNet + Transformer (with cross-attention to text)
    # "DownBlock2D" = ResNet only (no attention, used at the deepest level)
    down_block_types: Tuple[str, ...] = (
        "CrossAttnDownBlock2D",  # Level 0: 96x96 -> 48x48
        "CrossAttnDownBlock2D",  # Level 1: 48x48 -> 24x24
        "CrossAttnDownBlock2D",  # Level 2: 24x24 -> 12x12
        "DownBlock2D",           # Level 3: 12x12 (no downsample, deepest)
    )
    mid_block_type: str = "UNetMidBlock2DCrossAttn"
    up_block_types: Tuple[str, ...] = (
        "UpBlock2D",             # Level 3: 12x12 -> 24x24
        "CrossAttnUpBlock2D",    # Level 2: 24x24 -> 48x48
        "CrossAttnUpBlock2D",    # Level 1: 48x48 -> 96x96
        "CrossAttnUpBlock2D",    # Level 0: 96x96 (no upsample, final)
    )

    only_cross_attention: bool = False
    block_out_channels: Tuple[int, ...] = (320, 640, 1280, 1280)  # Channels per level
    layers_per_block: int = 2           # ResNet+Attn pairs per encoder level
    downsample_padding: int = 1
    mid_block_scale_factor: float = 1.0
    act_fn: str = "silu"                # Activation function throughout
    norm_num_groups: int = 32           # GroupNorm groups
    norm_eps: float = 1e-5
    cross_attention_dim: int = 1024     # OpenCLIP ViT-H embedding dimension
    attention_head_dim: Tuple[int, ...] = (5, 10, 20, 20)  # Per-level head dimensions
    use_linear_projection: bool = True  # Linear instead of Conv1x1 in Transformer

    class_embed_type: Optional[str] = None   # Not used in standard SD 2.x
    num_class_embeds: Optional[int] = None
    upcast_attention: bool = False            # Cast Q/K to float32 for stability
    resnet_time_scale_shift: str = "default"

## 2. Utility Functions

### Sinusoidal Timestep Embedding

The diffusion timestep (an integer 0–1000) needs to be converted into a
continuous vector the network can process. We use the same sinusoidal
positional encoding as the original Transformer paper:

$$\text{emb}_{2i} = \sin\!\left(\frac{t}{10000^{2i/d}}\right), \quad
  \text{emb}_{2i+1} = \cos\!\left(\frac{t}{10000^{2i/d}}\right)$$

SD 2.x sets `flip_sin_to_cos=True`, so the cos components come first.
This produces a 320-dim vector (matching `block_out_channels[0]`).

In [ ]:
def get_activation(name: str):
    """Return an activation module by name."""
    if name == "silu":
        return nn.SiLU()
    elif name == "gelu":
        return nn.GELU()
    else:
        raise ValueError(f"Unsupported activation: {name}")


def timestep_embedding(
    timesteps: torch.Tensor,
    dim: int,
    flip_sin_to_cos: bool = True,
    freq_shift: int = 0,
    max_period: int = 10000,
):
    """
    Create sinusoidal timestep embeddings.

    Args:
        timesteps: (batch,) integer timesteps
        dim: embedding dimension (320 for SD 2.x)
        flip_sin_to_cos: if True, output [cos, sin] instead of [sin, cos]
        freq_shift: shift the frequency spectrum (0 for SD 2.x)

    Returns:
        (batch, dim) sinusoidal embeddings
    """
    half = dim // 2
    # Compute log-spaced frequencies: from 1 to 1/max_period
    exponent = -math.log(max_period) * torch.arange(
        start=0, end=half, dtype=torch.float32, device=timesteps.device
    )
    exponent = exponent / (half - freq_shift)
    freqs = torch.exp(exponent)  # (half,)

    # Outer product: (batch, 1) * (1, half) -> (batch, half)
    args = timesteps.float()[:, None] * freqs[None]

    # Concatenate sin and cos
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

    if flip_sin_to_cos:
        # Swap halves: [sin, cos] -> [cos, sin]
        emb = torch.cat([emb[:, half:], emb[:, :half]], dim=-1)

    if dim % 2 == 1:
        emb = F.pad(emb, (0, 1))  # Pad if odd dimension

    return emb

## 3. Timestep Embedding MLP

The sinusoidal encoding (320-dim) is projected through a 2-layer MLP to
produce a richer 1280-dim representation (4× expansion, same as SD 1.5).
This gives the network learnable capacity to represent the noise schedule.

In [ ]:
class TimestepEmbedding(nn.Module):
    """
    Projects sinusoidal timestep encoding through a 2-layer MLP.
    320 -> 1280 -> 1280

    Same structure as SD 1.5's TimeEmbedding, but parameterized
    by config rather than hardcoded.
    """

    def __init__(self, in_dim: int, out_dim: int, act_fn: str = "silu"):
        super().__init__()
        self.linear_1 = nn.Linear(in_dim, out_dim)   # 320 -> 1280
        self.act = get_activation(act_fn)
        self.linear_2 = nn.Linear(out_dim, out_dim)  # 1280 -> 1280

    def forward(self, x):
        x = self.linear_1(x)
        x = self.act(x)
        x = self.linear_2(x)
        return x

## 4. ResNet Block with Timestep Conditioning

The backbone building block. Same idea as SD 1.5:

```
x -> GroupNorm -> SiLU -> Conv3x3 -> (+time_proj) -> GroupNorm -> SiLU -> Conv3x3 -> (+skip)
```

The timestep embedding (1280-dim) is projected to `out_channels` and **added**
to the feature map between the two convolutions. This tells the network what
noise level it's operating at.

In [ ]:
class ResnetBlock2D(nn.Module):
    """
    Residual block with timestep conditioning.

    Compared to SD 1.5's UNET_ResidualBlock, this uses the diffusers naming
    convention (norm1/norm2 vs groupnorm_feature/groupnorm_merged) and
    parameterizes GroupNorm groups and epsilon via config.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        temb_channels: int,  # Time embedding dim (1280)
        groups: int = 32,
        eps: float = 1e-5,
        act_fn: str = "silu",
    ):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels

        # First conv path
        self.norm1 = nn.GroupNorm(num_groups=groups, num_channels=in_channels, eps=eps, affine=True)
        self.act1 = get_activation(act_fn)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)

        # Timestep projection: 1280 -> out_channels
        self.time_emb_proj = nn.Linear(temb_channels, out_channels)

        # Second conv path
        self.norm2 = nn.GroupNorm(num_groups=groups, num_channels=out_channels, eps=eps, affine=True)
        self.act2 = get_activation(act_fn)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

        # Skip connection: 1x1 conv if channels change, identity otherwise
        if in_channels != out_channels:
            self.conv_shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        else:
            self.conv_shortcut = None

    def forward(self, x, temb):
        # x: (B, in_ch, H, W)
        # temb: (B, 1280) time embedding
        residual = x

        x = self.norm1(x)
        x = self.act1(x)
        x = self.conv1(x)

        # Add timestep: project (B, 1280) -> (B, out_ch) -> (B, out_ch, 1, 1)
        temb_proj = self.time_emb_proj(temb)[:, :, None, None]
        x = x + temb_proj

        x = self.norm2(x)
        x = self.act2(x)
        x = self.conv2(x)

        if self.conv_shortcut is not None:
            residual = self.conv_shortcut(residual)

        return x + residual

## 5. Attention Components

SD 2.x uses the same three-part attention block as SD 1.5:
1. **Self-Attention** - spatial features attend to each other
2. **Cross-Attention** - spatial features attend to text (OpenCLIP) tokens
3. **GeGLU FFN** - gated feed-forward network

### Key difference: Unified CrossAttention class

Unlike SD 1.5 (which had separate `SelfAttention` and `CrossAttention` classes),
SD 2.x / diffusers uses a **single `CrossAttention` class** for both. When
`context=None`, it becomes self-attention (Q, K, V all come from the same input).

### GeGLU (Gated Linear Unit with GELU)

The feed-forward network uses GeGLU instead of plain GELU:
$$\text{GeGLU}(x) = (xW_1) \odot \text{GELU}(xW_2)$$

where the linear layer outputs 2× the inner dimension, split into the value
and gate halves. This gating mechanism is more expressive than a simple
activation function.

In [ ]:
class GEGLU(nn.Module):
    """Gated Linear Unit with GELU activation.

    Projects input to 2x the output dimension, splits in half,
    and uses one half to gate the other via GELU.
    """

    def __init__(self, dim_in: int, dim_out: int):
        super().__init__()
        # Output is 2x because we split into value + gate
        self.proj = nn.Linear(dim_in, dim_out * 2)

    def forward(self, x):
        x, gate = self.proj(x).chunk(2, dim=-1)
        return x * F.gelu(gate)


class FeedForward(nn.Module):
    """
    GeGLU feed-forward network with 4x expansion.

    input (dim) -> GEGLU (dim -> 4*dim) -> Dropout -> Linear (4*dim -> dim)

    Note: Uses nn.ModuleList (not nn.Sequential) to match the diffusers
    weight naming convention: net.0, net.1, net.2
    """

    def __init__(self, dim: int, mult: int = 4, dropout: float = 0.0):
        super().__init__()
        inner_dim = dim * mult
        self.net = nn.ModuleList([
            GEGLU(dim, inner_dim),     # net.0: GeGLU projection
            nn.Dropout(dropout),       # net.1: Dropout (usually 0.0 at inference)
            nn.Linear(inner_dim, dim), # net.2: Output projection
        ])

    def forward(self, x):
        for layer in self.net:
            x = layer(x)
        return x


class CrossAttention(nn.Module):
    """
    Unified attention module: handles both self-attention and cross-attention.

    When context=None -> self-attention (Q, K, V from same input)
    When context=text  -> cross-attention (Q from latent, K/V from text)

    Differences from SD 1.5:
    - Single class for both self/cross attention
    - Q/K/V projections have bias=False (SD 1.5 varied)
    - Uses PyTorch's scaled_dot_product_attention (flash attention when available)
    - to_out is nn.ModuleList to match diffusers weight format
    """

    def __init__(
        self,
        query_dim: int,
        cross_attention_dim: Optional[int] = None,  # None = self-attention
        heads: int = 8,
        dim_head: int = 64,
        dropout: float = 0.0,
        upcast_attention: bool = False,
    ):
        super().__init__()
        inner_dim = heads * dim_head
        # For self-attention, K/V come from the same source as Q
        cross_attention_dim = cross_attention_dim if cross_attention_dim is not None else query_dim

        self.heads = heads
        self.dim_head = dim_head
        self.upcast_attention = upcast_attention

        # Q/K/V projections (no bias, unlike some SD 1.5 implementations)
        self.to_q = nn.Linear(query_dim, inner_dim, bias=False)
        self.to_k = nn.Linear(cross_attention_dim, inner_dim, bias=False)
        self.to_v = nn.Linear(cross_attention_dim, inner_dim, bias=False)

        # Output projection + dropout as ModuleList (matches diffusers naming)
        self.to_out = nn.ModuleList([
            nn.Linear(inner_dim, query_dim, bias=True),
            nn.Dropout(dropout),
        ])

    def forward(self, x, context=None):
        # x: (B, Seq_Q, Dim)  - spatial features (flattened H*W)
        # context: (B, Seq_KV, Dim_KV) - text embeddings, or None for self-attn
        if context is None:
            context = x  # Self-attention: K, V come from x itself

        q = self.to_q(x)
        k = self.to_k(context)
        v = self.to_v(context)

        b, n, _ = q.shape
        h = self.heads
        d = self.dim_head

        # Reshape for multi-head: (B, Seq, H*D) -> (B, H, Seq, D)
        q = q.view(b, n, h, d).transpose(1, 2)
        k = k.view(b, -1, h, d).transpose(1, 2)
        v = v.view(b, -1, h, d).transpose(1, 2)

        # Optional upcast for numerical stability (some SD 2.x configs use this)
        if self.upcast_attention:
            q = q.float()
            k = k.float()
            v = v.float()

        # Use PyTorch's optimized attention (uses Flash Attention when possible)
        x = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=None,
            dropout_p=0.0,
            is_causal=False,
        )

        # Reshape back: (B, H, Seq, D) -> (B, Seq, H*D)
        x = x.transpose(1, 2).contiguous().view(b, n, h * d)

        # Output projection + dropout
        x = self.to_out[0](x)
        x = self.to_out[1](x)
        return x

### Transformer Block and Spatial Transformer

The `BasicTransformerBlock` is the core unit: self-attention => cross-attention
=> GeGLU FFN, all with pre-norm and residual connections.

The `Transformer2DModel` wraps this to handle 2D feature maps:
1. GroupNorm the input feature map (B, C, H, W)
2. Flatten spatial dims to a sequence: (B, H*W, C)
3. Project with a Linear layer (SD 2.x) or Conv1x1 (SD 1.5)
4. Apply one or more BasicTransformerBlocks
5. Project back and reshape to (B, C, H, W)
6. Add residual connection

In [ ]:
class BasicTransformerBlock(nn.Module):
    """
    One transformer block inside the U-Net's Transformer2DModel.

    Contains three sub-blocks, all with pre-LayerNorm + residual:
    1. Self-Attention  (attn1, context=None)  - spatial coherence
    2. Cross-Attention (attn2, context=text)  - text conditioning
    3. GeGLU FFN       (ff)                   - feature transformation

    This is identical in structure to SD 1.5's UNET_AttentionBlock,
    but uses the unified CrossAttention class for both attention types.
    """

    def __init__(
        self,
        dim: int,
        num_heads: int,
        head_dim: int,
        cross_attention_dim: int,
        upcast_attention: bool = False,
    ):
        super().__init__()
        # Self-attention: cross_attention_dim=None means Q/K/V all from x
        self.norm1 = nn.LayerNorm(dim)
        self.attn1 = CrossAttention(
            query_dim=dim,
            cross_attention_dim=None,  # Self-attention
            heads=num_heads,
            dim_head=head_dim,
            upcast_attention=upcast_attention,
        )

        # Cross-attention: Q from latent, K/V from text
        self.norm2 = nn.LayerNorm(dim)
        self.attn2 = CrossAttention(
            query_dim=dim,
            cross_attention_dim=cross_attention_dim,  # 1024 for SD 2.x
            heads=num_heads,
            dim_head=head_dim,
            upcast_attention=upcast_attention,
        )

        # GeGLU feed-forward
        self.norm3 = nn.LayerNorm(dim)
        self.ff = FeedForward(dim)

    def forward(self, x, context):
        # Pre-norm + attention + residual for each sub-block
        x = x + self.attn1(self.norm1(x), context=None)     # Self-attention
        x = x + self.attn2(self.norm2(x), context=context)  # Cross-attention with text
        x = x + self.ff(self.norm3(x))                      # Feed-forward
        return x


class Transformer2DModel(nn.Module):
    """
    Spatial Transformer: wraps BasicTransformerBlock for 2D feature maps.

    Handles the (B, C, H, W) <-> (B, H*W, C) reshaping and optional
    linear/conv projection.

    SD 2.x uses use_linear_projection=True (Linear layers),
    SD 1.5 used Conv2d 1x1 (equivalent but different weight naming).
    """

    def __init__(
        self,
        in_channels: int,
        num_attention_heads: int,
        attention_head_dim: int,
        cross_attention_dim: int,
        num_layers: int = 1,
        norm_num_groups: int = 32,
        use_linear_projection: bool = True,
        upcast_attention: bool = False,
    ):
        super().__init__()
        inner_dim = num_attention_heads * attention_head_dim
        self.in_channels = in_channels
        self.norm = nn.GroupNorm(norm_num_groups, in_channels, eps=1e-6, affine=True)

        self.use_linear_projection = use_linear_projection
        if use_linear_projection:
            # SD 2.x: Linear projection (applied after spatial flatten)
            self.proj_in = nn.Linear(in_channels, inner_dim)
            self.proj_out = nn.Linear(inner_dim, in_channels)
        else:
            # SD 1.x: Conv2d 1x1 projection (applied before spatial flatten)
            self.proj_in = nn.Conv2d(in_channels, inner_dim, kernel_size=1)
            self.proj_out = nn.Conv2d(inner_dim, in_channels, kernel_size=1)

        self.transformer_blocks = nn.ModuleList([
            BasicTransformerBlock(
                dim=inner_dim,
                num_heads=num_attention_heads,
                head_dim=attention_head_dim,
                cross_attention_dim=cross_attention_dim,
                upcast_attention=upcast_attention,
            )
            for _ in range(num_layers)
        ])

    def forward(self, x, encoder_hidden_states):
        # x: (B, C, H, W)
        # encoder_hidden_states: (B, 77, 1024) text embeddings
        residual = x
        b, c, h, w = x.shape

        x = self.norm(x)

        if self.use_linear_projection:
            # SD 2.x path: flatten first, then linear project
            x = x.permute(0, 2, 3, 1).reshape(b, h * w, c)  # (B, H*W, C)
            x = self.proj_in(x)
            for block in self.transformer_blocks:
                x = block(x, encoder_hidden_states)
            x = self.proj_out(x)
            x = x.reshape(b, h, w, c).permute(0, 3, 1, 2)   # (B, C, H, W)
        else:
            # SD 1.x path: conv project first, then flatten
            x = self.proj_in(x)
            inner_dim = x.shape[1]
            x = x.permute(0, 2, 3, 1).reshape(b, h * w, inner_dim)
            for block in self.transformer_blocks:
                x = block(x, encoder_hidden_states)
            x = x.reshape(b, h, w, inner_dim).permute(0, 3, 1, 2)
            x = self.proj_out(x)

        return x + residual

## 6. Sampling Helpers

Simple modules for spatial resolution changes:
- **Downsample2D**: Strided 3×3 convolution (halves spatial dims)
- **Upsample2D**: Nearest-neighbor interpolation + 3×3 convolution (doubles spatial dims)

These are the same as SD 1.5, just extracted into reusable classes.

In [ ]:
class Downsample2D(nn.Module):
    """Halve spatial resolution with a strided 3x3 convolution."""

    def __init__(self, channels: int, padding: int = 1):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, stride=2, padding=padding)

    def forward(self, x):
        return self.conv(x)


class Upsample2D(nn.Module):
    """Double spatial resolution with nearest interpolation + 3x3 convolution."""

    def __init__(self, channels: int):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2.0, mode="nearest")
        x = self.conv(x)
        return x

## 7. Encoder and Decoder Blocks

The U-Net encoder and decoder are built from four types of blocks:

| Block Type | Contains | Used For |
||||
| `DownBlock2D` | ResNets only | Deepest encoder level (no text attention needed) |
| `CrossAttnDownBlock2D` | ResNets + Transformers | Encoder levels that need text conditioning |
| `UpBlock2D` | ResNets + skip concat | Deepest decoder level |
| `CrossAttnUpBlock2D` | ResNets + Transformers + skip concat | Decoder levels with text conditioning |

### Skip Connections
Each encoder block saves its intermediate outputs. The decoder blocks receive
these via concatenation (doubling the channel count, which the ResNet handles
via its `conv_shortcut`). This preserves fine-grained spatial detail that would
otherwise be lost during downsampling.

### Decoder has layers_per_block + 1 layers
The decoder has one extra ResNet+Attention pair per level compared to the encoder
(`layers_per_block + 1`). This is because skip connections are consumed one at a
time, and the downsample output also needs to be handled.

In [ ]:
class DownBlock2D(nn.Module):
    """
    Encoder block WITHOUT attention (used at the deepest level).

    Contains `num_layers` ResNet blocks + optional downsample.
    Returns both the output and all intermediate states (for skip connections).
    """

    def __init__(
        self, in_channels, out_channels, temb_channels, num_layers,
        add_downsample=True, groups=32, eps=1e-5, act_fn="silu",
        downsample_padding=1,
    ):
        super().__init__()
        self.resnets = nn.ModuleList()
        for i in range(num_layers):
            ch_in = in_channels if i == 0 else out_channels
            self.resnets.append(
                ResnetBlock2D(ch_in, out_channels, temb_channels, groups, eps, act_fn)
            )
        self.downsamplers = nn.ModuleList(
            [Downsample2D(out_channels, padding=downsample_padding)] if add_downsample else []
        )

    def forward(self, x, temb):
        outputs = []  # Collect states for skip connections
        for resnet in self.resnets:
            x = resnet(x, temb)
            outputs.append(x)
        for down in self.downsamplers:
            x = down(x)
            outputs.append(x)  # Downsampled state is also a skip connection
        return x, outputs


class CrossAttnDownBlock2D(nn.Module):
    """
    Encoder block WITH attention (ResNet + Transformer at each layer).

    Each layer applies: ResNet(x, time) -> Transformer(x, text) -> save skip
    Then optionally downsamples.
    """

    def __init__(
        self, in_channels, out_channels, temb_channels, num_layers,
        cross_attention_dim, num_attention_heads, attention_head_dim,
        add_downsample=True, groups=32, eps=1e-5, act_fn="silu",
        downsample_padding=1, use_linear_projection=True, upcast_attention=False,
    ):
        super().__init__()
        self.resnets = nn.ModuleList()
        self.attentions = nn.ModuleList()

        for i in range(num_layers):
            ch_in = in_channels if i == 0 else out_channels
            self.resnets.append(
                ResnetBlock2D(ch_in, out_channels, temb_channels, groups, eps, act_fn)
            )
            self.attentions.append(
                Transformer2DModel(
                    in_channels=out_channels,
                    num_attention_heads=num_attention_heads,
                    attention_head_dim=attention_head_dim,
                    cross_attention_dim=cross_attention_dim,
                    num_layers=1,
                    norm_num_groups=groups,
                    use_linear_projection=use_linear_projection,
                    upcast_attention=upcast_attention,
                )
            )

        self.downsamplers = nn.ModuleList(
            [Downsample2D(out_channels, padding=downsample_padding)] if add_downsample else []
        )

    def forward(self, x, temb, encoder_hidden_states):
        outputs = []
        for resnet, attn in zip(self.resnets, self.attentions):
            x = resnet(x, temb)                    # Timestep conditioning
            x = attn(x, encoder_hidden_states)     # Text conditioning
            outputs.append(x)
        for down in self.downsamplers:
            x = down(x)
            outputs.append(x)
        return x, outputs


class UpBlock2D(nn.Module):
    """
    Decoder block WITHOUT attention.

    Each ResNet receives the concatenation of the current features and
    a skip connection from the encoder (popped from the stack in reverse order).
    """

    def __init__(
        self, in_channels, prev_output_channel, out_channels, temb_channels,
        num_layers, add_upsample=True, groups=32, eps=1e-5, act_fn="silu",
    ):
        super().__init__()
        self.resnets = nn.ModuleList()

        for i in range(num_layers):
            # Channel math accounts for skip connection concatenation
            res_skip_channels = in_channels if i == num_layers - 1 else out_channels
            resnet_in = prev_output_channel if i == 0 else out_channels
            self.resnets.append(
                ResnetBlock2D(
                    resnet_in + res_skip_channels,  # Concat doubles channels
                    out_channels, temb_channels, groups, eps, act_fn,
                )
            )

        self.upsamplers = nn.ModuleList(
            [Upsample2D(out_channels)] if add_upsample else []
        )

    def forward(self, x, res_hidden_states_tuple, temb):
        for resnet in self.resnets:
            skip = res_hidden_states_tuple[-1]           # Pop last skip connection
            res_hidden_states_tuple = res_hidden_states_tuple[:-1]
            x = torch.cat([x, skip], dim=1)             # Concat along channel dim
            x = resnet(x, temb)

        for up in self.upsamplers:
            x = up(x)
        return x


class CrossAttnUpBlock2D(nn.Module):
    """
    Decoder block WITH attention.

    Same as UpBlock2D but each ResNet is followed by a Transformer
    (self-attention + cross-attention with text).
    """

    def __init__(
        self, in_channels, prev_output_channel, out_channels, temb_channels,
        num_layers, cross_attention_dim, num_attention_heads, attention_head_dim,
        add_upsample=True, groups=32, eps=1e-5, act_fn="silu",
        use_linear_projection=True, upcast_attention=False,
    ):
        super().__init__()
        self.resnets = nn.ModuleList()
        self.attentions = nn.ModuleList()

        for i in range(num_layers):
            res_skip_channels = in_channels if i == num_layers - 1 else out_channels
            resnet_in = prev_output_channel if i == 0 else out_channels
            self.resnets.append(
                ResnetBlock2D(
                    resnet_in + res_skip_channels,
                    out_channels, temb_channels, groups, eps, act_fn,
                )
            )
            self.attentions.append(
                Transformer2DModel(
                    in_channels=out_channels,
                    num_attention_heads=num_attention_heads,
                    attention_head_dim=attention_head_dim,
                    cross_attention_dim=cross_attention_dim,
                    num_layers=1,
                    norm_num_groups=groups,
                    use_linear_projection=use_linear_projection,
                    upcast_attention=upcast_attention,
                )
            )

        self.upsamplers = nn.ModuleList(
            [Upsample2D(out_channels)] if add_upsample else []
        )

    def forward(self, x, res_hidden_states_tuple, temb, encoder_hidden_states):
        for resnet, attn in zip(self.resnets, self.attentions):
            skip = res_hidden_states_tuple[-1]
            res_hidden_states_tuple = res_hidden_states_tuple[:-1]
            x = torch.cat([x, skip], dim=1)
            x = resnet(x, temb)
            x = attn(x, encoder_hidden_states)

        for up in self.upsamplers:
            x = up(x)
        return x

## 8. Mid Block (Bottleneck)

The bottleneck sits between the encoder and decoder at the lowest spatial
resolution (12×12 for SD 2.x at 768×768). It contains:

```
ResNet -> Transformer (self-attn + cross-attn) -> ResNet
```

This is the deepest point where global context (via attention over the
smallest feature map) and text conditioning are applied.

In [ ]:
class UNetMidBlock2DCrossAttn(nn.Module):
    """
    Bottleneck block: ResNet -> Transformer -> ResNet.

    At the deepest spatial resolution, this block applies the most
    expensive attention operations (smallest spatial size = cheapest
    to attend over all positions).

    Note: Uses nn.ModuleList (not direct attributes) to match the
    diffusers weight naming: resnets.0, attentions.0, resnets.1
    """

    def __init__(
        self, in_channels, temb_channels, cross_attention_dim,
        num_attention_heads, attention_head_dim,
        groups=32, eps=1e-5, act_fn="silu",
        use_linear_projection=True, upcast_attention=False,
    ):
        super().__init__()

        self.resnets = nn.ModuleList([
            ResnetBlock2D(in_channels, in_channels, temb_channels, groups, eps, act_fn),
            ResnetBlock2D(in_channels, in_channels, temb_channels, groups, eps, act_fn),
        ])

        self.attentions = nn.ModuleList([
            Transformer2DModel(
                in_channels=in_channels,
                num_attention_heads=num_attention_heads,
                attention_head_dim=attention_head_dim,
                cross_attention_dim=cross_attention_dim,
                num_layers=1,
                norm_num_groups=groups,
                use_linear_projection=use_linear_projection,
                upcast_attention=upcast_attention,
            )
        ])

    def forward(self, x, temb, encoder_hidden_states):
        x = self.resnets[0](x, temb)
        x = self.attentions[0](x, encoder_hidden_states)
        x = self.resnets[1](x, temb)
        return x

## 9. Full UNet2DConditionModel

This assembles all the pieces into the complete U-Net, driven entirely by the
`UNet2DConditionConfig` dataclass.

### Architecture overview (SD 2.1-base)

```
Input: (B, 4, 96, 96) noisy latent
  │
  ├─ conv_in: 4 -> 320
  │
  ├─ Down Block 0 (CrossAttn): 320ch, 96x96 -> 48x48  [2 ResNet+Attn + Downsample]
  ├─ Down Block 1 (CrossAttn): 640ch, 48x48 -> 24x24  [2 ResNet+Attn + Downsample]
  ├─ Down Block 2 (CrossAttn): 1280ch, 24x24 -> 12x12 [2 ResNet+Attn + Downsample]
  ├─ Down Block 3 (Plain):     1280ch, 12x12          [2 ResNet, no downsample]
  │
  ├─ Mid Block: 1280ch, 12x12                         [ResNet + Attn + ResNet]
  │
  ├─ Up Block 0 (Plain):       1280ch, 12x12 -> 24x24 [3 ResNet + Upsample]
  ├─ Up Block 1 (CrossAttn):   1280ch, 24x24 -> 48x48 [3 ResNet+Attn + Upsample]
  ├─ Up Block 2 (CrossAttn):   640ch, 48x48 -> 96x96  [3 ResNet+Attn + Upsample]
  ├─ Up Block 3 (CrossAttn):   320ch, 96x96           [3 ResNet+Attn, no upsample]
  │
  ├─ GroupNorm -> SiLU -> conv_out: 320 -> 4
  │
Output: (B, 4, 96, 96) predicted noise
```

The config-driven design means this same class works for SD 2.0, 2.1,
and 2.1-base - only the config values differ.

In [ ]:
class UNet2DConditionModel(nn.Module):
    """
    The complete SD 2.x U-Net: config-driven noise prediction network.

    ~865M parameters (similar to SD 1.5), but with:
    - Larger cross-attention dimension (1024 vs 768)
    - Variable attention head dimensions per level
    - Linear projections in Transformer2DModel
    """

    def __init__(self, config: UNet2DConditionConfig):
        super().__init__()
        self.config = config

        # Time embedding dimension: 4x the base channel count
        time_embed_dim = config.block_out_channels[0] * 4  # 320 * 4 = 1280
        self.time_proj_dim = config.block_out_channels[0]  # 320

        # Input convolution: 4 latent channels -> 320
        self.conv_in = nn.Conv2d(
            config.in_channels, config.block_out_channels[0],
            kernel_size=3, padding=1,
        )

        # Timestep MLP: 320 -> 1280
        self.time_embedding = TimestepEmbedding(
            self.time_proj_dim, time_embed_dim, act_fn=config.act_fn,
        )

        # ===== Encoder (Down Blocks) =====
        self.down_blocks = nn.ModuleList()
        output_channel = config.block_out_channels[0]

        for i, down_block_type in enumerate(config.down_block_types):
            input_channel = output_channel
            output_channel = config.block_out_channels[i]
            is_final = i == len(config.block_out_channels) - 1

            # Compute number of attention heads from per-level head dim
            head_dim = config.attention_head_dim[i]
            num_heads = output_channel // head_dim

            if down_block_type == "CrossAttnDownBlock2D":
                block = CrossAttnDownBlock2D(
                    in_channels=input_channel,
                    out_channels=output_channel,
                    temb_channels=time_embed_dim,
                    num_layers=config.layers_per_block,
                    cross_attention_dim=config.cross_attention_dim,
                    num_attention_heads=num_heads,
                    attention_head_dim=head_dim,
                    add_downsample=not is_final,  # No downsample at deepest level
                    groups=config.norm_num_groups,
                    eps=config.norm_eps,
                    act_fn=config.act_fn,
                    downsample_padding=config.downsample_padding,
                    use_linear_projection=config.use_linear_projection,
                    upcast_attention=config.upcast_attention,
                )
            elif down_block_type == "DownBlock2D":
                block = DownBlock2D(
                    in_channels=input_channel,
                    out_channels=output_channel,
                    temb_channels=time_embed_dim,
                    num_layers=config.layers_per_block,
                    add_downsample=not is_final,
                    groups=config.norm_num_groups,
                    eps=config.norm_eps,
                    act_fn=config.act_fn,
                    downsample_padding=config.downsample_padding,
                )
            else:
                raise ValueError(f"Unsupported down block: {down_block_type}")

            self.down_blocks.append(block)

        # ===== Bottleneck (Mid Block) =====
        mid_channels = config.block_out_channels[-1]  # 1280
        mid_head_dim = config.attention_head_dim[-1]    # 20
        mid_num_heads = mid_channels // mid_head_dim    # 64

        self.mid_block = UNetMidBlock2DCrossAttn(
            in_channels=mid_channels,
            temb_channels=time_embed_dim,
            cross_attention_dim=config.cross_attention_dim,
            num_attention_heads=mid_num_heads,
            attention_head_dim=mid_head_dim,
            groups=config.norm_num_groups,
            eps=config.norm_eps,
            act_fn=config.act_fn,
            use_linear_projection=config.use_linear_projection,
            upcast_attention=config.upcast_attention,
        )

        # ===== Decoder (Up Blocks) =====
        self.up_blocks = nn.ModuleList()
        reversed_block_out_channels = list(reversed(config.block_out_channels))
        reversed_attention_head_dim = list(reversed(config.attention_head_dim))
        prev_output_channel = reversed_block_out_channels[0]

        for i, up_block_type in enumerate(config.up_block_types):
            output_channel = reversed_block_out_channels[i]
            input_channel = reversed_block_out_channels[
                min(i + 1, len(reversed_block_out_channels) - 1)
            ]
            is_final = i == len(config.up_block_types) - 1

            head_dim = reversed_attention_head_dim[i]
            num_heads = output_channel // head_dim

            if up_block_type == "CrossAttnUpBlock2D":
                block = CrossAttnUpBlock2D(
                    in_channels=input_channel,
                    prev_output_channel=prev_output_channel,
                    out_channels=output_channel,
                    temb_channels=time_embed_dim,
                    num_layers=config.layers_per_block + 1,  # +1 for skip handling
                    cross_attention_dim=config.cross_attention_dim,
                    num_attention_heads=num_heads,
                    attention_head_dim=head_dim,
                    add_upsample=not is_final,
                    groups=config.norm_num_groups,
                    eps=config.norm_eps,
                    act_fn=config.act_fn,
                    use_linear_projection=config.use_linear_projection,
                    upcast_attention=config.upcast_attention,
                )
            elif up_block_type == "UpBlock2D":
                block = UpBlock2D(
                    in_channels=input_channel,
                    prev_output_channel=prev_output_channel,
                    out_channels=output_channel,
                    temb_channels=time_embed_dim,
                    num_layers=config.layers_per_block + 1,
                    add_upsample=not is_final,
                    groups=config.norm_num_groups,
                    eps=config.norm_eps,
                    act_fn=config.act_fn,
                )
            else:
                raise ValueError(f"Unsupported up block: {up_block_type}")

            self.up_blocks.append(block)
            prev_output_channel = output_channel

        # ===== Output =====
        self.conv_norm_out = nn.GroupNorm(
            num_groups=config.norm_num_groups,
            num_channels=config.block_out_channels[0],
            eps=config.norm_eps, affine=True,
        )
        self.conv_act = get_activation(config.act_fn)
        self.conv_out = nn.Conv2d(
            config.block_out_channels[0], config.out_channels,
            kernel_size=3, padding=1,
        )

    def forward(
        self,
        sample: torch.Tensor,           # (B, 4, 96, 96) noisy latent
        timestep: torch.Tensor,          # (B,) or scalar
        encoder_hidden_states: torch.Tensor,  # (B, 77, 1024) text embeddings
    ):
        if sample.ndim != 4:
            raise ValueError("sample must be (B, C, H, W)")

        # Handle scalar or single-element timestep
        if timestep.ndim == 0:
            timestep = timestep[None]
        if timestep.ndim == 1 and timestep.shape[0] == 1 and sample.shape[0] > 1:
            timestep = timestep.expand(sample.shape[0])

        # 1. Time embedding: scalar -> sinusoidal -> MLP
        t_emb = timestep_embedding(
            timestep, self.time_proj_dim,
            flip_sin_to_cos=self.config.flip_sin_to_cos,
            freq_shift=self.config.freq_shift,
        )
        emb = self.time_embedding(t_emb)  # (B, 1280)

        # 2. Input projection
        x = self.conv_in(sample)  # (B, 320, 96, 96)

        # 3. Encoder path (save skip connections)
        down_block_res_samples = [x]
        for block in self.down_blocks:
            if isinstance(block, CrossAttnDownBlock2D):
                x, res_samples = block(x, emb, encoder_hidden_states)
            else:
                x, res_samples = block(x, emb)
            down_block_res_samples.extend(res_samples)

        # 4. Bottleneck
        x = self.mid_block(x, emb, encoder_hidden_states)

        # 5. Decoder path (consume skip connections in reverse)
        for block in self.up_blocks:
            n_res = len(block.resnets)
            res_samples = tuple(down_block_res_samples[-n_res:])
            down_block_res_samples = down_block_res_samples[:-n_res]

            if isinstance(block, CrossAttnUpBlock2D):
                x = block(x, res_samples, emb, encoder_hidden_states)
            else:
                x = block(x, res_samples, emb)

        # 6. Output projection
        x = self.conv_norm_out(x)
        x = self.conv_act(x)
        x = self.conv_out(x)  # (B, 4, 96, 96)

        return x  # Predicted noise

## 10. Loading Weights from HuggingFace

Unlike SD 1.5 (which shipped as a single `.ckpt` file requiring manual key
mapping), SD 2.x uses the **diffusers** format: each component (UNet, VAE, CLIP)
is stored separately with matching parameter names.

Since our class names and structure match diffusers exactly, we can load weights
directly with `load_state_dict()` - no manual key mapping needed!

We try safetensors first (faster, safer) and fall back to PyTorch `.bin`.

In [ ]:
def load_unet_config_from_hf(repo_id: str, subfolder: str = "unet") -> UNet2DConditionConfig:
    """
    Download and parse the UNet config.json from a HuggingFace diffusers repo.

    This reads the JSON config shipped alongside the weights and maps it
    to our UNet2DConditionConfig dataclass.
    """
    config_path = hf_hub_download(repo_id=repo_id, filename=f"{subfolder}/config.json")
    with open(config_path, "r") as f:
        cfg = json.load(f)

    return UNet2DConditionConfig(
        sample_size=cfg["sample_size"],
        in_channels=cfg["in_channels"],
        out_channels=cfg["out_channels"],
        center_input_sample=cfg.get("center_input_sample", False),
        flip_sin_to_cos=cfg.get("flip_sin_to_cos", True),
        freq_shift=cfg.get("freq_shift", 0),
        down_block_types=tuple(cfg["down_block_types"]),
        mid_block_type=cfg.get("mid_block_type", "UNetMidBlock2DCrossAttn"),
        up_block_types=tuple(cfg["up_block_types"]),
        only_cross_attention=cfg.get("only_cross_attention", False),
        block_out_channels=tuple(cfg["block_out_channels"]),
        layers_per_block=cfg["layers_per_block"],
        downsample_padding=cfg.get("downsample_padding", 1),
        mid_block_scale_factor=cfg.get("mid_block_scale_factor", 1.0),
        act_fn=cfg.get("act_fn", "silu"),
        norm_num_groups=cfg.get("norm_num_groups", 32),
        norm_eps=cfg.get("norm_eps", 1e-5),
        cross_attention_dim=cfg["cross_attention_dim"],
        attention_head_dim=(
            tuple(cfg["attention_head_dim"]) if isinstance(cfg["attention_head_dim"], list)
            else tuple([cfg["attention_head_dim"]] * len(cfg["block_out_channels"]))
        ),
        use_linear_projection=cfg.get("use_linear_projection", True),
        class_embed_type=cfg.get("class_embed_type", None),
        num_class_embeds=cfg.get("num_class_embeds", None),
        upcast_attention=cfg.get("upcast_attention", False),
        resnet_time_scale_shift=cfg.get("resnet_time_scale_shift", "default"),
    )


def load_unet_weights_from_hf(model: nn.Module, repo_id: str, subfolder: str = "unet"):
    """
    Download and load UNet weights from a HuggingFace diffusers repo.

    Tries safetensors first (faster, safer), falls back to pytorch .bin.
    Since our architecture matches diffusers naming, no key remapping is needed.
    """
    state_dict = None

    # Try safetensors first
    try:
        weights_path = hf_hub_download(
            repo_id=repo_id,
            filename=f"{subfolder}/diffusion_pytorch_model.safetensors"
        )
        state_dict = load_file(weights_path)
    except Exception:
        pass

    # Fall back to .bin
    if state_dict is None:
        weights_path = hf_hub_download(
            repo_id=repo_id,
            filename=f"{subfolder}/diffusion_pytorch_model.bin"
        )
        state_dict = torch.load(weights_path, map_location="cpu")

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"Missing keys ({len(missing)}): {missing[:5]}...")
    if unexpected:
        print(f"Unexpected keys ({len(unexpected)}): {unexpected[:5]}...")
    if not missing and not unexpected:
        print("All weights loaded successfully!")

    return model

## 11. Putting It All Together

Let's load the SD 2.1-base U-Net from HuggingFace and run a forward pass
with dummy inputs to verify everything works.

**Note**: This only loads the U-Net. A full generation pipeline would also
need the OpenCLIP text encoder, VAE encoder/decoder, and a sampler
(similar to what we built for SD 1.5).

In [ ]:
# Load config and weights from HuggingFace 
repo_id = "sd2-community/stable-diffusion-2-1"

print("Loading config...")
config = load_unet_config_from_hf(repo_id, subfolder="unet")
print(f"  Sample size: {config.sample_size}")
print(f"  Block channels: {config.block_out_channels}")
print(f"  Cross-attention dim: {config.cross_attention_dim}")
print(f"  Attention head dims: {config.attention_head_dim}")

print("\nBuilding model...")
model = UNet2DConditionModel(config)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"  Total parameters: {total_params:,} ({total_params / 1e6:.0f}M)")

print("\nLoading weights...")
model = load_unet_weights_from_hf(model, repo_id, subfolder="unet")
model.eval()

In [ ]:
# Forward pass with dummy inputs 
# SD 2.1-base: 96x96 latent, 77 text tokens at 1024-dim

batch = 1
latent = torch.randn(batch, 4, 96, 96)            # Noisy latent from VAE
timestep = torch.tensor([999], dtype=torch.long)  # High noise level
text_context = torch.randn(batch, 77, 1024)       # OpenCLIP text embeddings

# Run on GPU with mixed precision for speed
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.half().to(device)
latent = latent.half().to(device)
text_context = text_context.half().to(device)

print(f"Running forward pass on {device}...")
with torch.inference_mode(), torch.autocast(device, dtype=torch.float16):
    out = model(latent, timestep.to(device), text_context)

print(f"Input shape:  {latent.shape}")
print(f"Output shape: {out.shape}")
print(f"Output range: [{out.min().item():.3f}, {out.max().item():.3f}]")
print("\nThe output is the predicted noise (same shape as input latent).")

## Summary: SD 1.5 vs SD 2.x U-Net Architecture

| Aspect | SD 1.5 (our previous tutorial) | SD 2.x (this tutorial) |
||||
| **Architecture definition** | Hardcoded in class constructors | Config-driven via dataclass |
| **Attention class** | Separate `SelfAttention` + `CrossAttention` | Unified `CrossAttention` (context=None => self) |
| **Attention implementation** | Manual Q@K/sqrt(d)@V | `F.scaled_dot_product_attention` (Flash Attn) |
| **Projection in Transformer** | Conv2d 1×1 | Linear (use_linear_projection=True) |
| **Q/K/V bias** | Mixed (True/False depending on block) | No bias (bias=False) |
| **FeedForward** | Inline in attention block | Separate class with nn.ModuleList |
| **Weight loading** | Manual 900+ key mapping (`weights.py`) | Direct `load_state_dict` (matching names) |
| **Cross-attention dim** | 768 (CLIP ViT-L/14) | 1024 (OpenCLIP ViT-H/14) |
| **Latent size** | 64×64 (512px image) | 96×96 (768px image) |

The fundamental algorithm is identical: a U-Net with skip connections that
predicts noise conditioned on text (via cross-attention) and timestep
(via addition after MLP). The differences are in engineering choices
(config-driven, unified attention, Flash Attention support) and scale
(larger text encoder, higher resolution).

## 12. Full Generation Pipeline

Now we wire together all four SD 2.1-base components into a runnable txt2img
(and img2img) pipeline - the same structure as SD 1.5, adapted for the
larger latent space and stronger text encoder.

```
  Text prompt
      │
      V
  OpenCLIP ViT-H/14  => context (B, 77, 1024)
      │
      V
  Pure noise z_T  (B, 4, 96, 96)        ← 768×768 image => 96×96 latent
      │
      V  loop T => 0  (DDIM, 30 steps)
  UNet2DConditionModel => predicted noise ε̂
      │
      V
  Clean latent z_0  (B, 4, 96, 96)
      │
      V
  VAE Decoder  => RGB image  (B, 3, 768, 768)
```

### SD 2.1 vs SD 1.5 - Pipeline Differences

| Step | SD 1.5 | SD 2.1 |
|----|----|----|
| Text encoder | CLIP ViT-L/14 (768-dim) | **OpenCLIP ViT-H/14 (1024-dim)** |
| Latent shape | (4, 64, 64) | **(4, 96, 96)** |
| Native resolution | 512×512 | **768×768** |
| Weight format | Single `.ckpt` + key-map | **Diffusers per-component safetensors** |
| Sampler | DDPM (1000 steps) | **DDIM (30 steps, deterministic)** |

We use `diffusers` `AutoencoderKL` and `CLIPTextModel` for the VAE and text
encoder - they ship in exactly the format our U-Net was designed for, so no
key remapping is needed.

### 12a. DDIM Sampler

DDIM (Denoising Diffusion Implicit Models, Song et al. 2020) re-uses the same
DDPM noise schedule as training but takes large deterministic steps, enabling
high-quality generation in 20–50 steps instead of 1000.

**DDIM update rule** (one step: t => t_prev):
```
  x̂_0    = (x_t − √(1−ᾱ_t) · ε̂) / √ᾱ_t     ← predicted clean latent
  x_{t-1} = √ᾱ_{t-1} · x̂_0 + √(1−ᾱ_{t-1}) · ε̂   ← deterministic step
```
Setting `eta=0` (default) makes the process fully deterministic: same seed => same image.

In [ ]:
import numpy as np
from tqdm import tqdm
from PIL import Image

# It does not work as DDIMScheduler from diffusers
# Need to rewrite to have an exact match to the original implementation
# but for now it is fine to have as an example
class DDIMSampler:
    """
    DDIM (Denoising Diffusion Implicit Models) sampler.

    Uses the SD 2.1 scaled-linear beta schedule but skips steps, converging
    in 20-50 forward passes instead of 1000.

    Reference: Song et al. 2020  https://arxiv.org/abs/2010.02502

    Parameters
    -
    num_train_timesteps : int
        Training steps (SD 2.1 = 1000).
    beta_start / beta_end : float
        Scaled-linear schedule identical to SD 1.5.
    eta : float
        0.0 = deterministic DDIM; 1.0 = stochastic DDPM-equivalent.
    """

    def __init__(
        self,
        num_train_timesteps: int = 1000,
        beta_start: float = 0.00085,
        beta_end: float = 0.012,
        eta: float = 0.0,
    ):
        # Scaled-linear beta schedule (same as SD 1.5 / 2.1)
        self.betas = torch.linspace(
            beta_start ** 0.5, beta_end ** 0.5, num_train_timesteps
        ).pow(2)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)  # ᾱ_t
        self.num_train_timesteps = num_train_timesteps
        self.eta = eta
        self.timesteps = None
        self.num_inference_steps = None

    def set_timesteps(self, num_inference_steps: int):
        """Choose evenly-spaced timesteps, high => low (reversed for denoising)."""
        step = self.num_train_timesteps // num_inference_steps
        ts = np.arange(0, self.num_train_timesteps, step)[::-1].copy()
        self.timesteps = torch.from_numpy(ts.astype(np.int64))
        self.num_inference_steps = num_inference_steps

    def step(
        self,
        timestep: int,
        latents: torch.Tensor,
        noise_pred: torch.Tensor,
        generator: torch.Generator = None,
    ) -> torch.Tensor:
        """
        One DDIM denoising step: x_t => x_{t-1}.

        Parameters
        -
        timestep : int
            Current timestep (0..999).
        latents : Tensor (B, 4, H, W)
            Current noisy latent x_t.
        noise_pred : Tensor (B, 4, H, W)
            U-Net predicted noise ε̂.

        Returns
        -
        Tensor (B, 4, H, W) - x_{t-1}
        """
        t = timestep
        step_size = self.num_train_timesteps // self.num_inference_steps
        prev_t = t - step_size

        alpha_prod_t    = self.alphas_cumprod[t]
        alpha_prod_prev = self.alphas_cumprod[prev_t] if prev_t >= 0 else torch.tensor(1.0)

        # Predicted clean latent:  x_0 = (x_t − sqrt(1−a_t)*eps) / sqrt(a_t)
        pred_x0 = (latents - (1 - alpha_prod_t).sqrt() * noise_pred) / alpha_prod_t.sqrt()
        pred_x0 = pred_x0.clamp(-1, 1)

        # Variance for stochastic term (0 when eta=0)
        sigma = self.eta * (
            (1 - alpha_prod_prev) / (1 - alpha_prod_t) *
            (1 - alpha_prod_t / alpha_prod_prev)
        ).clamp(min=0).sqrt()

        dir_xt = (1 - alpha_prod_prev - sigma ** 2).clamp(min=0).sqrt() * noise_pred

        x_prev = alpha_prod_prev.sqrt() * pred_x0 + dir_xt
        if self.eta > 0 and generator is not None:
            noise = torch.randn(
                latents.shape, generator=generator,
                device=latents.device, dtype=latents.dtype,
            )
            x_prev = x_prev + sigma * noise

        return x_prev

    def add_noise(
        self,
        latents: torch.Tensor,
        timestep: int,
        generator: torch.Generator = None,
    ) -> torch.Tensor:
        """
        Forward diffusion (for img2img): add noise at a given timestep.
        x_t = sqrt(a_t) · x_0 + sqrt(1 - a_t) · eps
        """
        alpha_prod = self.alphas_cumprod[timestep]
        noise = torch.randn(
            latents.shape, generator=generator,
            device=latents.device, dtype=latents.dtype,
        )
        return alpha_prod.sqrt() * latents + (1 - alpha_prod).sqrt() * noise

### 12b. Load All Pipeline Components

We load from `sd2-community/stable-diffusion-2-1` (native 768×768):

| Component | Source | Purpose |
|----|----|----|
| `vae` | `AutoencoderKL` (diffusers) | Encode images => 4-ch latents; decode latents => images |
| `text_encoder` | `CLIPTextModel` (OpenCLIP ViT-H/14) | Text => 1024-dim sequence (77 tokens) |
| `tokenizer` | `CLIPTokenizer` | BPE tokenizer, max 77 tokens |
| `model` (U-Net) | `UNet2DConditionModel` | Already built and loaded above |

In [ ]:
from diffusers import AutoencoderKL, UNet2DConditionModel, DDIMScheduler
from transformers import CLIPTextModel, CLIPTokenizer

REPO_ID = "sd2-community/stable-diffusion-2-1"

# VAE
# 4-channel latents, 8× spatial compression: 768px => 96×96 latent
print("Loading VAE...")
vae = AutoencoderKL.from_pretrained(REPO_ID, subfolder="vae")
vae.eval()

# Text encoder (OpenCLIP ViT-H/14)
# Cross-attention dim in SD 2.1 is 1024 (vs 768 in SD 1.5)
print("Loading text encoder (OpenCLIP ViT-H/14)...")
text_encoder = CLIPTextModel.from_pretrained(REPO_ID, subfolder="text_encoder")
text_encoder.eval()

# Tokenizer
tokenizer = CLIPTokenizer.from_pretrained(REPO_ID, subfolder="tokenizer")

# U-Net (HuggingFace UNet2DConditionModel)
# The custom UNet built above is for teaching the architecture.
# For actual inference we use HF's implementation, which applies attention
# at properly downsampled resolutions and avoids the O(N²) OOM that the
# from-scratch version suffers at 96×96 full-resolution self-attention.
# The implementation above takes about at least 40Gb..
print("Loading U-Net (UNet2DConditionModel)...")
unet_hf = UNet2DConditionModel.from_pretrained(REPO_ID, subfolder="unet")
unet_hf.eval()

# Shadow the tutorial `model` variable so generate_sd2() picks up the
# efficient HF UNet without any other code changes.
model = unet_hf

# Constants
VAE_SCALE = 0.18215   # SD 1.5 and 2.x use the same VAE scaling factor
WIDTH, HEIGHT = 768, 768
LATENT_H, LATENT_W = HEIGHT // 8, WIDTH // 8  # 96, 96

print(f"\nAll components loaded.")
print(f"  U-Net      : {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params")
print(f"  Resolution : {WIDTH}×{HEIGHT}")
print(f"  Latent     : {LATENT_H}×{LATENT_W}")
print(f"  Text dim   : 1024 (OpenCLIP ViT-H/14)")

# Scheduler
# We use diffusers' DDIMScheduler instead of the hand-rolled DDIMSampler
# above.  The custom sampler used  prev_t = t - (T // steps)  which is
# only correct for a uniform schedule; the scaled-linear beta schedule
# used by SD 2.x is non-uniform, so that formula produces wrong alpha
# values.  DDIMScheduler handles the indexing correctly.
from diffusers.utils.torch_utils import randn_tensor
scheduler = DDIMScheduler.from_pretrained(REPO_ID, subfolder="scheduler")
print("  Scheduler  : DDIMScheduler loaded")

### 12c. Generate Function

The generate function mirrors the SD 1.5 pipeline from our first tutorial,
adapted for SD 2.1's larger latent space and text embeddings.

**Classifier-Free Guidance (CFG)** is unchanged:

$$\hat{\epsilon}_{guided} = \hat{\epsilon}_{uncond} + s \cdot (\hat{\epsilon}_{cond} - \hat{\epsilon}_{uncond})$$

Higher `cfg_scale` => stronger prompt adherence (typical: 7–9).

In [ ]:
def encode_prompt_sd2(prompt: str, neg_prompt: str, tokenizer, text_encoder, device):
    """
    Encode a positive and negative prompt with OpenCLIP ViT-H/14.

    Returns a batched (2, 77, 1024) tensor - [conditional, unconditional] -
    ready for classifier-free guidance.
    """
    def _enc(text):
        ids = tokenizer(
            text, padding="max_length",
            max_length=tokenizer.model_max_length,
            truncation=True, return_tensors="pt",
        ).input_ids.to(device)
        return text_encoder(ids).last_hidden_state  # (1, 77, 1024)

    return torch.cat([_enc(prompt), _enc(neg_prompt)])  # (2, 77, 1024)


def decode_latents_sd2(vae, latents: torch.Tensor) -> np.ndarray:
    """
    Decode (B, 4, H/8, W/8) latents to (B, H, W, 3) uint8 images.
    Reverses the 0.18215 VAE scaling factor applied during encoding.
    """
    latents = latents / VAE_SCALE
    images = vae.decode(latents).sample         # (B, 3, H, W) in [-1, 1]
    images = (images / 2 + 0.5).clamp(0, 1)     # -> [0, 1]
    images = (images * 255).round().to(torch.uint8)
    return images.permute(0, 2, 3, 1).cpu().numpy()  # (B, H, W, 3)


def generate_sd2(
    prompt: str,
    negative_prompt: str = "",
    input_image=None,               # PIL Image for img2img; None = txt2img
    strength: float = 0.8,          # img2img noise level (0=no change, 1=full)
    cfg_scale: float = 7.5,         # CFG guidance scale
    num_inference_steps: int = 30,
    seed: int = 42,
    device: str = "cuda",
) -> Image.Image:
    """
    Full SD 2.1-base txt2img / img2img generation pipeline.

    Uses diffusers\'s DDIMScheduler for correct timestep indexing on the
    scaled-linear beta schedule (the hand-rolled DDIMSampler used a uniform
    prev_t = t - T//steps formula which gives wrong alphas here).
    """
    generator = torch.Generator(device=device).manual_seed(seed)

    # Step 1: Encode text
    text_encoder.to(device)
    with torch.no_grad():
        context = encode_prompt_sd2(
            prompt, negative_prompt, tokenizer, text_encoder, device
        )  # (2, 77, 1024)
    text_encoder.to("cpu")

    # Step 2: Scheduler timesteps
    scheduler.set_timesteps(num_inference_steps, device=device)

    # Step 3: Initial latent
    if input_image is not None:
        # img2img: encode => scale => add noise up to start timestep
        vae.to(device)
        img = input_image.convert("RGB").resize((WIDTH, HEIGHT))
        img_t = (
            torch.tensor(np.array(img), dtype=torch.float32, device=device)
            .permute(2, 0, 1).unsqueeze(0) / 127.5 - 1.0
        )  # (1, 3, H, W) in [-1, 1]
        with torch.no_grad():
            latents = vae.encode(img_t).latent_dist.sample(generator=generator)
            latents = latents * VAE_SCALE
        vae.to("cpu")

        start_step = int(num_inference_steps * (1 - strength))
        timesteps  = scheduler.timesteps[start_step:]
        noise = randn_tensor(
            latents.shape, generator=generator, device=device, dtype=latents.dtype
        )
        latents = scheduler.add_noise(latents, noise, timesteps[:1])
    else:
        # txt2img: start from pure Gaussian noise
        latents = randn_tensor(
            (1, 4, LATENT_H, LATENT_W),
            generator=generator, device=device, dtype=torch.float32,
        )
        timesteps = scheduler.timesteps

    # Step 4: Denoising loop
    unet = model.to(device)
    context = context.to(device)
    latents = latents.to(device)

    autocast_ctx = (
        torch.autocast(device_type="cuda", dtype=torch.float16)
        if device == "cuda"
        else torch.autocast(device_type="cpu", dtype=torch.bfloat16)
    )

    with torch.no_grad(), autocast_ctx:
        for t in tqdm(timesteps, desc="SD 2.1 denoising"):
            latent_in = latents.repeat(2, 1, 1, 1)  # (2, 4, 96, 96) for CFG

            # Broadcast scalar timestep to a (2,) long tensor
            timestep_batch = torch.tensor([t] * 2, device=device, dtype=torch.long)

            noise_pred = unet(
                latent_in,
                timestep_batch,
                encoder_hidden_states=context,
            ).sample  # (2, 4, 96, 96)

            # Classifier-Free Guidance
            noise_cond, noise_uncond = noise_pred.chunk(2)
            noise_pred = noise_uncond + cfg_scale * (noise_cond - noise_uncond)

            # Scheduler step - handles correct alpha/sigma indexing internally
            latents = scheduler.step(
                noise_pred, t, latents, generator=generator
            ).prev_sample

    unet.to("cpu")

    # Step 5: Decode latent => image
    vae.to(device)
    with torch.no_grad():
        images = decode_latents_sd2(vae, latents.float())
    vae.to("cpu")

    return Image.fromarray(images[0])

### 12d. Text-to-Image Inference

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

output = generate_sd2(
    prompt="A majestic castle on a rocky cliff at sunset, dramatic golden light, "
           "8k resolution, highly detailed, trending on artstation",
    negative_prompt="blurry, low quality, artifacts, deformed, ugly",
    cfg_scale=7.5,
    num_inference_steps=30,
    seed=42,
    device=DEVICE,
)

output.save("sd2_txt2img.png")
print(f"Saved: sd2_txt2img.png  (size={output.size})")
output

### 12e. Image-to-Image Inference

img2img encodes an input image into the latent space, adds noise to a level
determined by `strength`, then denoises with the text prompt:

- `strength=1.0` - pure txt2img (input image ignored)
- `strength=0.75` - significant transformation, structure preserved
- `strength=0.3` - light retouching

In [ ]:
!wget -q https://upload.wikimedia.org/wikipedia/commons/thumb/7/74/A-Cat.jpg/1280px-A-Cat.jpg -O cat_sd2.jpg

In [ ]:
from IPython.display import display

input_image = Image.open("cat_sd2.jpg").convert("RGB")

output_img2img = generate_sd2(
    prompt="A cat wearing round glasses, highly detailed, ultra sharp, cinematic, 100mm lens, 8k resolution.",
    negative_prompt="",
    input_image=input_image,
    strength=0.75,
    cfg_scale=7.5,
    num_inference_steps=30,
    seed=42,
    device=DEVICE,
)

output_img2img.save("sd2_img2img.png")
print("Input (resized):")
display(input_image.resize((384, 384)))
print("\nOutput (img2img):")
display(output_img2img.resize((384, 384)))